# 02. Gold Layer Feature Fusion & Multi-Modal Integration

**Upstream Dependencies:** `data/silver/outlet_master.csv`, `data/silver/transactions_history_final.csv`, `data/silver/outlet_spatial_decay_features.csv`  
**Downstream Target:** `data/gold/master_feature_table.csv`

---

## Mission Overview & Architectural Context
In Phase 1, the pipeline established an unconstrained baseline using flat catchment counts. For this Final Round, we are transforming those preliminary pipelines into an enterprise-grade decision engine focused on **Potential-Based Allocation**. 

This notebook governs the compilation of our **Gold Master Feature Table**. It collapses millions of granular historical transaction records into static store-level behavior profiles, integrates our advanced, non-linear distance-decay spatial matrices, and deploys rigorous data validation guardrails.

### Key Engineering Transformations:
1. **Behavioral Stream Aggregation:** Collapsing raw, time-series transactions into localized velocity, consistency, and lifetime revenue profiles.
2. **Multi-Modal Feature Fusion:** Performing a deterministic relational left-join with our off-line engineered continuous spatial decay vectors.
3. **Data Hygiene Quarantine & Imputation:** Isolating the 40 coordinate-quarantined retail nodes and deploying global median statistical imputation to eliminate `NaN` propagation prior to machine learning training.

In [6]:
import os
import sys
import numpy as np
import pandas as pd
from pathlib import Path

# Establish robust relative project pathing to enable seamless execution across workspaces
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SILVER_DIR = PROJECT_ROOT / "data" / "silver"
GOLD_DIR = PROJECT_ROOT / "data" / "gold"

print(" Gold Layer Pipeline Environment Initialized.")
print(f"  -> Ingesting from Silver Storage: {SILVER_DIR.resolve()}")
print(f"  -> Promoting to Gold Storage:     {GOLD_DIR.resolve()}")

 Gold Layer Pipeline Environment Initialized.
  -> Ingesting from Silver Storage: /Users/judithfernando/DataStorm7/data/silver
  -> Promoting to Gold Storage:     /Users/judithfernando/DataStorm7/data/gold


## 1. Upstream Data Synchronization & Validation Guardrails
Prior to executing structural joins, we load our clean core internal layers and our newly engineered spatial decay asset. We enforce explicit pre-execution checks to guarantee file presence and prevent silent pipeline compilation failures.

In [8]:
# Define mandatory file tracking paths
master_path = SILVER_DIR / "outlet_master.csv"
tx_path = SILVER_DIR / "transactions_history_final.csv"
spatial_decay_path = SILVER_DIR / "outlet_spatial_decay_features.csv"

# Absolute validation check before loading datasets into memory
for path in [master_path, tx_path, spatial_decay_path]:
    if not path.exists():
        raise FileNotFoundError(f" Critical Pipeline Failure: Upstream asset missing -> {path}")

# Ingest dataframes
df_master = pd.read_csv(master_path)
df_tx = pd.read_csv(tx_path)
df_spatial = pd.read_csv(spatial_decay_path)

print(" Upstream components successfully synchronized:")
print(f"  -> Base Outlet Master Matrix:       {df_master.shape[0]} unique locations")
print(f"  -> Cleaned Historical Transactions: {df_tx.shape[0]} transaction vectors")
print(f"  -> Continuous Spatial Decay Map:    {df_spatial.shape[0]} processed spatial nodes")

 Upstream components successfully synchronized:
  -> Base Outlet Master Matrix:       20000 unique locations
  -> Cleaned Historical Transactions: 2339409 transaction vectors
  -> Continuous Spatial Decay Map:    19960 processed spatial nodes


## 2. Behavioral Feature Extraction (Time-Series Aggregation)
Machine learning models evaluating potential capacity require an invariant matrix structure mapping exactly **one row per unique store identifier**. 

To collapse Member 1's time-series transaction registry without losing variance or historical demand signals, we group records by `Outlet_ID` to capture long-term performance baseline profiles:
* `lifetime_volume_liters`: Measures absolute historical baseline volumetric consumption.
* `lifetime_revenue_lkr`: Evaluates the raw economic throughput footprint of the outlet.
* `avg_transaction_size_liters`: Quantifies localized logistics velocity and ordering scale.
* `active_transaction_months`: Proxies baseline brand loyalty and market consistency.

In [9]:
print(" Processing transaction streams into static behavioral summaries...")

# Group time-series files to synthesize unique outlet business profiles
df_tx_profiles = df_tx.groupby('Outlet_ID').agg(
    lifetime_volume_liters=('Volume_Liters', 'sum'),
    lifetime_revenue_lkr=('Total_Bill_Value', 'sum'),
    avg_transaction_size_liters=('Volume_Liters', 'mean'),
    active_transaction_months=('Month', 'count')
).reset_index()

print("🔗 Executing structural left-merge with core master characteristics...")
# Perform an explicit Left-Join anchored securely to the master framework
df_gold_base = pd.merge(df_master, df_tx_profiles, on='Outlet_ID', how='left')

# Business Logic Check: Onboarded outlets with zero transaction history must register 0, not NaN
behavioral_cols = [
    'lifetime_volume_liters', 'lifetime_revenue_lkr', 
    'avg_transaction_size_liters', 'active_transaction_months'
]
df_gold_base[behavioral_cols] = df_gold_base[behavioral_cols].fillna(0)

print(f" Behavioral Profiling Matrix complete. Footprint: {df_gold_base.shape}")

 Processing transaction streams into static behavioral summaries...
🔗 Executing structural left-merge with core master characteristics...
 Behavioral Profiling Matrix complete. Footprint: (20000, 11)


## 3. Multi-Modal Fusion & Spatial-Relational Mapping
We now integrate the advanced, non-linear geospatial layer generated via our vectorized distance-decay pipeline (`src/features/build_spatial_decay.py`). 

This step integrates continuous environmental features directly into our training dataset:
1. **Category-Specific Decay Scores:** Continuous demand pull indices calculated for schools, transit options, and local commercial clusters using an exponential decay function ($e^{-\frac{\text{distance}}{\text{scale}}}$).
2. **Overall Spatial Gravity Score:** An all-inclusive geographical catchment density vector encompassing all 22,870 scraped OpenStreetMap objects.
3. **Competitor Catchment Density:** Internal competitor clustering profiles calculating market saturation boundaries within an immediate 500m radius.

In [10]:
print(" Fusing advanced spatial characteristics with structural behavioral profile matrix...")

# Relational join utilizing unique Outlet_ID as the immutable primary key index
df_gold_merged = pd.merge(df_gold_base, df_spatial, on='Outlet_ID', how='left')

print(f" Geospatial alignment complete. Intermediary dimensions: {df_gold_merged.shape}")

 Fusing advanced spatial characteristics with structural behavioral profile matrix...
 Geospatial alignment complete. Intermediary dimensions: (20000, 16)


## 4. Fault-Recovery Quality Assurance (Median Imputation Guardrail)
During the data forensics pipeline executed in the Silver layer, **40 unique outlets** were systematically quarantined due to corrupted or unresolvable raw coordinate strings. To prevent row dropping and preserve the complete evaluation footprint of **exactly 20,000 outlets**, these locations were carried forward into the master framework.

**The Problem:** Because these 40 outlets lack valid coordinates, they could not be mapped to map points or competitive stores, resulting in blank `NaN` cells across our new advanced spatial columns. If passed directly to algorithms like XGBoost or LightGBM, these missing cells can break compilation boundaries.

**The Solution:** We deploy a dynamic, non-biasing statistical global median imputation guardrail. This safely immunizes missing cell boundaries without shifting the spatial population distributions or injecting data leaks.

In [11]:
print(" Initializing statistical fault-recovery guardrails over quarantined spatial features...")

# Specify newly generated advanced round 2 feature space vectors
spatial_feature_cols = [
    'school_decay_score', 'transit_decay_score', 
    'commercial_decay_score', 'overall_spatial_gravity_score', 
    'competitor_catchment_density'
]

# Track metrics for data audit records
imputed_cell_count = 0

for col in spatial_feature_cols:
    null_mask = df_gold_merged[col].isna()
    if null_mask.any():
        null_sum = null_mask.sum()
        imputed_cell_count += null_sum
        
        # Calculate global population median for the feature space
        median_value = df_gold_merged[col].median()
        
        # Apply imputation mapping directly
        df_gold_merged[col] = df_gold_merged[col].fillna(median_value)
        print(f"  -> Imputed {null_sum} missing cells in column '{col}' with population median: {median_value:.4f}")

print(f" Fault-Recovery Audit: Successfully neutralized {imputed_cell_count} missing cells across quarantined records.")

 Initializing statistical fault-recovery guardrails over quarantined spatial features...
  -> Imputed 40 missing cells in column 'school_decay_score' with population median: 0.0000
  -> Imputed 40 missing cells in column 'transit_decay_score' with population median: 0.0000
  -> Imputed 40 missing cells in column 'commercial_decay_score' with population median: 0.0030
  -> Imputed 40 missing cells in column 'overall_spatial_gravity_score' with population median: 0.0809
  -> Imputed 40 missing cells in column 'competitor_catchment_density' with population median: 2.0000
 Fault-Recovery Audit: Successfully neutralized 200 missing cells across quarantined records.


## 5. Enterprise Unit Testing & Promotion to Gold Status
Prior to exporting our master dataset to the machine learning modeling environment managed by Member 3, we execute strict enterprise unit constraints. The dataset must pass all quality checks to guarantee total compliance with Round 2 structural standards.

In [12]:
print("🧪 Commencing production unit testing matrix...")

# Test 1: Preservation of the complete retail population footprint (Strictly 20,000 unique records) [cite: 31]
assert len(df_gold_merged) == 20000, f" CRITICAL ASSERTION FAILURE: Row dimensions modified! Current: {len(df_gold_merged)}"
print("  [PASS] Unit Test 1: Strict population constraint verified at 20,000 rows.")

# Test 2: Integrity of unique identifier primary key mapping
assert df_gold_merged['Outlet_ID'].isna().sum() == 0, " CRITICAL ASSERTION FAILURE: Missing primary keys detected!"
print("  [PASS] Unit Test 2: Primary key completeness validated (0 missing Outlet_ID).")

# Test 3: Duplication and index collision verification
assert df_gold_merged['Outlet_ID'].duplicated().sum() == 0, " CRITICAL ASSERTION FAILURE: Primary key index collision detected!"
print("  [PASS] Unit Test 3: Primary key uniqueness validated (0 duplicate Outlet_ID).")

# Test 4: Absolute global completeness verification sweep
global_nans = df_gold_merged.isna().sum().sum()
assert global_nans == 0, f" CRITICAL ASSERTION FAILURE: {global_nans} unhandled missing cells found in dataset matrix!"
print("  [PASS] Unit Test 4: Global completion check validated (0 missing cells inside matrix).")

# --- PROMOTION TO GOLD MASTER STORAGE ---
GOLD_DIR.mkdir(parents=True, exist_ok=True)
master_output_file = GOLD_DIR / "master_feature_table.csv"
df_gold_merged.to_csv(master_output_file, index=False)

print("\n GOLD LAYER FUSION COMPLETE! MASTER MODELING ASSET PROMOTED SUCCESSFULLY.")
print(f"Destination Target Location:  {master_output_file.resolve()}")
print(f"Final Matrix Geometry Shape:  {df_gold_merged.shape[0]} unique outlet entries x {df_gold_merged.shape[1]} clean model-ready columns.")

print("\n Senior Leadership Preview Matrix:")
display(df_gold_merged.head(3))

🧪 Commencing production unit testing matrix...
  [PASS] Unit Test 1: Strict population constraint verified at 20,000 rows.
  [PASS] Unit Test 2: Primary key completeness validated (0 missing Outlet_ID).
  [PASS] Unit Test 3: Primary key uniqueness validated (0 duplicate Outlet_ID).
  [PASS] Unit Test 4: Global completion check validated (0 missing cells inside matrix).

 GOLD LAYER FUSION COMPLETE! MASTER MODELING ASSET PROMOTED SUCCESSFULLY.
Destination Target Location:  /Users/judithfernando/DataStorm7/data/gold/master_feature_table.csv
Final Matrix Geometry Shape:  20000 unique outlet entries x 16 clean model-ready columns.

 Senior Leadership Preview Matrix:


,Outlet_ID,Outlet_Size,Cooler_Count,Outlet_Type,outlet_size_status,coord_status,has_valid_coord,lifetime_volume_liters,lifetime_revenue_lkr,avg_transaction_size_liters,active_transaction_months,school_decay_score,transit_decay_score,commercial_decay_score,overall_spatial_gravity_score,competitor_catchment_density
0,OUT_00001,Medium,1,Grocery,provided,valid,True,12287.669769,3.152152e+06,64.671946,190,1.061274e-02,0.001311,2.306505,2.955716,4.0
1,OUT_00002,Small,0,Hotel,provided,valid,True,14494.419585,3.703930e+06,71.754552,202,6.382290e-03,0.000002,0.100502,0.183306,5.0
2,OUT_00003,Small,1,Pharmacy,provided,valid,True,12620.115111,3.515750e+06,60.673630,208,2.114998e-07,0.000267,0.005715,0.003288,3.0
